# ML-Enhanced Screened Coulomb Interaction Approach for PC-Targeting Aptamer Design

**Author:** AbuHuwaiz Al-Rashidi  
**ORCID:** [0000-0002-6615-0664](https://orcid.org/0000-0002-6615-0664)  
**Affiliation:** King Saud University, Riyadh, Saudi Arabia  
**Reference SCIA paper:** Alharbi et al. (2026) [doi:10.1016/j.medidd.2026.100247](https://doi.org/10.1016/j.medidd.2026.100247)  
**License:** MIT

---

### Pipeline overview

| Step | Description |
|------|-------------|
| 1 | Install dependencies |
| 2 | SCI energy calculator — physics reimplementation |
| 3 | Dataset loading and feature engineering (31-D) |
| 4 | ML ensemble training with LOO-CV and bootstrapped 95% CI |
| 5 | Feature importance — three-method analysis |
| 6 | Virtual library screening — 50,000 DNA + RNA candidates |
| 7 | Negative control / decoy discrimination test |
| 8 | PCA — chemical space coverage |
| 9 | Publication figures (Figures 1–5) |

> **How to run:** Go to **Runtime → Run all** (Ctrl+F9). Total runtime ≈ 3–5 minutes.


In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────
!pip install numpy scipy scikit-learn matplotlib seaborn pandas --quiet
print('All dependencies installed.')

In [ ]:
# ── CELL 2: Imports and global settings ─────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.integrate import trapezoid
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from math import log2
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})
BLUE='#1F3864'; TEAL='#1D9E75'; AMBER='#BA7517'; RED='#E24B4A'
PURPLE='#7F77DD'; GRAY='#888780'; LGRAY='#F2F5FB'
print('Imports complete. numpy', np.__version__, '| sklearn', __import__('sklearn').__version__)

In [ ]:
# ── CELL 3: SCI Energy Reimplementation ─────────────────────────────────

"""
Open-source Python reimplementation of the Screened Coulomb Interaction
Approach (SCIA) from Alharbi et al. (2026) doi:10.1016/j.medidd.2026.100247
"""

# Physical constants
E0  = 8.854e-12  # Vacuum permittivity (F/m)
ER  = 80.0       # Relative permittivity water
E   = 1.6e-19    # Elementary charge (C)
KB  = 1.38e-23   # Boltzmann constant (J/K)
T_K = 310.0      # Physiological temperature (K)
N   = 1.0e26     # Ionic number density (m^-3)
PC_CHARGE = 2    # PC zwitterion net charge magnitude

# Nucleotide parameters — Table 1, Alharbi et al. (2026)
FUNCTIONAL_CHARGE = {'A': 3, 'G': 4, 'C': 4, 'T': 3, 'U': 3}
LATTICE_SIZE      = {'A': 0.34, 'G': 0.34, 'C': 0.30, 'T': 0.30, 'U': 0.30}

def sci_energy(abb_charge, r_nm_array, k_points=8000):
    """Screened Coulomb binding energy between ABB (charge=abb_charge) and PC."""
    r = r_nm_array * 1e-9
    k = np.linspace(1e9, 1e11, k_points)
    f = 1.0 / (2 * np.pi * KB * T_K * N)  # screening factor
    V_direct   = (abb_charge * PC_CHARGE * E**2) / (E0 * ER * k**2)
    V_screened = V_direct * np.exp(-f * k**2)
    return np.array([trapezoid(V_screened * np.cos(k * ri), k) for ri in r])

r_vals = np.linspace(0.1, 5.0, 300)
SCI_PROFILES = {nt: sci_energy(FUNCTIONAL_CHARGE[nt], r_vals) for nt in FUNCTIONAL_CHARGE}

# Verify rank ordering
e_at_rmin = {nt: abs(SCI_PROFILES[nt][0]) for nt in SCI_PROFILES}
rank_order = sorted(e_at_rmin, key=lambda x: -e_at_rmin[x])
print("SCI rank order:", " > ".join([f"{nt}(q={FUNCTIONAL_CHARGE[nt]})"
                                      for nt in rank_order]))
print("Expected (Alharbi et al. Fig 3): G > C > A > T ≈ U")
expected = set(rank_order[:2]) == {'G','C'} and set(rank_order[2:]) == {'A','T','U'}
print("Verification:", "PASSED ✓" if expected else "FAILED ✗")

# Pre-compute minimum-distance SCI energies for use as ML features
SCI_FEAT_VALS = {nt: sci_energy(FUNCTIONAL_CHARGE[nt], np.array([0.1]))[0]
                 for nt in "AGCT"}

In [ ]:
# ── CELL 4: Dataset — 18 PC aptamers + 2 negative controls ──────────────────

APTAMERS = [
    # (ID, sequence, binding_score, category)
    # Universal PC aptamers
    ("PCAPt1",  "GAAAAGGAGC", 1.000, "Universal"),
    ("PCAPt2",  "GAAAAGGATC", 0.975, "Universal"),
    # DNA-specific
    ("PCAPt3",  "GAAAAGGATG", 0.950, "DNA"),
    ("PCAPt4",  "GAAAGGATGC", 0.925, "DNA"),
    ("PCAPt5",  "GAAGGATGCA", 0.900, "DNA"),
    ("PCAPt6",  "GAAGGATGCG", 0.875, "DNA"),
    # RNA-specific
    ("PCAPt7",  "GAAAAGGAUC", 0.960, "RNA"),
    ("PCAPt8",  "GAAAAGGUAC", 0.930, "RNA"),
    ("PCAPt9",  "GAAGGUACGC", 0.900, "RNA"),
    ("PCAPt10", "GAAGGUACGG", 0.870, "RNA"),
    # Hybrid
    ("PCAPt11", "GAGCGGAUCC", 0.855, "Hybrid"),
    ("PCAPt12", "GAGCGGAUCG", 0.840, "Hybrid"),
    ("PCAPt13", "GAGCGGATCC", 0.820, "Hybrid"),
    ("PCAPt14", "GAGCGGATCG", 0.800, "Hybrid"),
    ("PCAPt15", "GAGCGGAUCU", 0.780, "Hybrid"),
    ("PCAPt16", "GAGUGGAUCC", 0.760, "Hybrid"),
    ("PCAPt17", "GCGCGGAUCC", 0.740, "Hybrid"),
    ("PCAPt18", "GCGCGGATCC", 0.720, "Hybrid"),
    # Negative controls
    ("NEG1",    "GGCGGCGGCG", 0.100, "Negative"),
    ("NEG2",    "AGGGTTTTTT", 0.080, "Negative"),
]

labels = [r[0] for r in APTAMERS]
seqs   = [r[1] for r in APTAMERS]
y      = np.array([r[2] for r in APTAMERS])
cats   = [r[3] for r in APTAMERS]

df_train = pd.DataFrame(APTAMERS, columns=["id","sequence","binding_score","category"])
print(df_train.to_string(index=False))

In [ ]:
# ── CELL 5: Feature Engineering — 31-dimensional vectors ────────────────────

DINUCLEOTIDES = [a+b for a in "AGCT" for b in "AGCT"]  # 16 dinucs

FEATURE_NAMES = (
    ["fA","fG","fC","fT","fU","GC","purine","charge_total","charge_dens",
     "charge_per_nm","length_nm","entropy","seq_len"]
    + DINUCLEOTIDES
    + ["SCI_A","SCI_G","SCI_C","SCI_T"]
)
print(f"Feature vector dimensionality: {len(FEATURE_NAMES)}")

def featurize(seq):
    seq = seq.upper(); n = len(seq)
    cnt = {nt: seq.count(nt) for nt in "AGCTU"}
    frac = {nt: cnt[nt]/n for nt in "AGCTU"}
    gc = (cnt["G"]+cnt["C"])/n
    pur = (cnt["A"]+cnt["G"])/n
    ch_tot = sum(FUNCTIONAL_CHARGE.get(nt,3) for nt in seq)
    ln = sum(LATTICE_SIZE.get(nt,0.32) for nt in seq)
    ch_dn = ch_tot/n; ch_nm = ch_tot/ln if ln>0 else 0
    ent = -sum((cnt[nt]/n)*log2(cnt[nt]/n) for nt in "AGCTU" if cnt[nt]>0)
    sdna = seq.replace("U","T")
    di = {d:0 for d in DINUCLEOTIDES}
    for i in range(len(sdna)-1):
        d=sdna[i:i+2]
        if d in di: di[d]+=1
    td = max(1,len(sdna)-1)
    dv = [di[d]/td for d in DINUCLEOTIDES]
    sf = [SCI_FEAT_VALS[nt] for nt in "AGCT"]
    return ([frac["A"],frac["G"],frac["C"],frac["T"],frac["U"],
             gc,pur,ch_tot,ch_dn,ch_nm,ln,ent,n] + dv + sf)

X_raw = np.array([featurize(s) for s in seqs])
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
print(f"Design matrix shape: {X.shape}  (n_samples x n_features)")

In [ ]:
# ── CELL 6: ML Ensemble — LOO-CV + bootstrapped 95% CI ──────────────────────

rf  = RandomForestRegressor(n_estimators=500, max_features="sqrt",
                             min_samples_leaf=2, random_state=42)
gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                 max_depth=2, random_state=42)
loo = LeaveOneOut()

# LOO-CV predictions
y_pred_rf  = cross_val_predict(rf,  X, y, cv=loo)
y_pred_gbr = cross_val_predict(gbr, X, y, cv=loo)

# Bootstrap 95% CI (B=2000)
def bootstrap_ci(y_true, y_pred, metric_fn, B=2000):
    rng = np.random.default_rng(42)
    vals = []
    for _ in range(B):
        idx = rng.integers(0,len(y_true),size=len(y_true))
        yb,pb = y_true[idx],y_pred[idx]
        if len(np.unique(yb))<2: continue
        vals.append(metric_fn(yb,pb))
    return np.percentile(vals,[2.5,97.5])

print("="*60)
print("LOO-CV RESULTS WITH BOOTSTRAPPED 95% CI  (B = 2,000)")
print("="*60)
for name, pred in [("RF", y_pred_rf), ("GBR", y_pred_gbr)]:
    r2  = r2_score(y, pred)
    mae = mean_absolute_error(y, pred)
    ci_r2  = bootstrap_ci(y, pred, r2_score)
    ci_mae = bootstrap_ci(y, pred, mean_absolute_error)
    print(f"\n{name}:")
    print(f"  R\u00b2  = {r2:.4f}  (95% CI: {ci_r2[0]:.4f} \u2013 {ci_r2[1]:.4f})")
    print(f"  MAE = {mae:.4f}  (95% CI: {ci_mae[0]:.4f} \u2013 {ci_mae[1]:.4f})")

# Fit on full dataset for downstream use
rf.fit(X,y); gbr.fit(X,y)
print("\nModels fitted on full dataset.")

In [ ]:
# ── CELL 7: Feature importance — three-method analysis ──────────────────────

pi_res = permutation_importance(rf, X, y, n_repeats=100, random_state=42)

feat_df = pd.DataFrame({
    "feature":       FEATURE_NAMES,
    "MDI_importance":rf.feature_importances_,
    "perm_mean":     pi_res.importances_mean,
    "perm_std":      pi_res.importances_std,
    "pearson_r":     [stats.pearsonr(X_raw[:,i],y)[0] for i in range(X_raw.shape[1])],
    "spearman_r":    [stats.spearmanr(X_raw[:,i],y)[0] for i in range(X_raw.shape[1])],
}).sort_values("perm_mean", ascending=False)

print("TOP 15 FEATURES — permutation importance (100 repeats)")
print(feat_df.head(15)[["feature","perm_mean","perm_std","pearson_r","spearman_r"]].to_string(index=False))

In [ ]:
# ── CELL 8: Virtual library screening — 50,000 sequences ────────────────────

def gen_lib(bases, n=25000, length=10, seed=42):
    rng = np.random.default_rng(seed)
    seqs = set()
    while len(seqs) < n:
        seqs.add("".join(rng.choice(list(bases), length)))
    return list(seqs)

dna_lib = gen_lib("AGCT", 25000)
rna_lib = gen_lib("AGCU", 25000, seed=43)
full_lib = dna_lib + rna_lib
lib_types = ["DNA"]*len(dna_lib) + ["RNA"]*len(rna_lib)

# Exclude training sequences
tr_set = set(seqs)
pairs = [(s,t) for s,t in zip(full_lib,lib_types) if s not in tr_set]
full_lib  = [p[0] for p in pairs]
lib_types = [p[1] for p in pairs]

print(f"Library size: {len(full_lib):,} sequences")

# Featurize + score
X_lib_raw = np.array([featurize(s) for s in full_lib])
X_lib     = scaler.transform(X_lib_raw)
s_rf  = rf.predict(X_lib)
s_gbr = gbr.predict(X_lib)
s_ens = (s_rf + s_gbr) / 2.0

# Structural fitness
def struct_fit(seq):
    seq = seq.upper().replace("U","T"); n = len(seq)
    gc = (seq.count("G")+seq.count("C"))/n
    rc = seq[::-1].translate(str.maketrans("ATGC","TACG"))
    sc = sum(a==b for a,b in zip(seq,rc))/n
    return gc*(1-0.5*sc)

ss  = np.array([struct_fit(s) for s in full_lib])
comp = 0.6*s_ens + 0.4*ss

# Hamming filter (min distance >= 2 from all training seqs)
def hamming(a,b):
    a=a.upper().replace("U","T"); b=b.upper().replace("U","T")
    return sum(x!=y for x,y in zip(a,b)) if len(a)==len(b) else 999

print("Applying diversity filter...")
top_mask = (s_ens > 0.85) & np.array(
    [all(hamming(full_lib[i], t) >= 2 for t in seqs) for i in range(len(full_lib))])
top_idx = np.where(top_mask)[0]
top_sorted = top_idx[np.argsort(-comp[top_idx])]

print(f"\nSCREENING SUMMARY")
print(f"  Total screened          = {len(full_lib):,}")
print(f"  High-affinity (>0.85)   = {(s_ens>0.85).sum():,}  ({(s_ens>0.85).mean()*100:.2f}%)")
print(f"  High-affinity + diverse = {len(top_idx):,}")
print(f"  Top candidate           = {full_lib[top_sorted[0]]} "
      f"(ens={s_ens[top_sorted[0]]:.4f}, composite={comp[top_sorted[0]]:.4f})")

print("\nTOP 10 NOVEL CANDIDATES:")
top10_data = []
for rank,i in enumerate(top_sorted[:10],1):
    top10_data.append({"rank":rank,"sequence":full_lib[i],"type":lib_types[i],
                       "ens_score":round(s_ens[i],4),"struct":round(ss[i],3),
                       "composite":round(comp[i],4)})
df_top10 = pd.DataFrame(top10_data)
print(df_top10.to_string(index=False))

In [ ]:
# ── CELL 9: Negative control / decoy discrimination ─────────────────────────

pos_scores  = y_pred_gbr[:18]           # 18 known PC aptamers
decoy_scores = s_ens[s_ens < 0.5][:200] # 200 random low-scoring library seqs

mwu_stat, mwu_p = stats.mannwhitneyu(pos_scores, decoy_scores, alternative="greater")
ks_stat,  ks_p  = stats.ks_2samp(pos_scores, decoy_scores)
effect_r = 1 - (2*mwu_stat)/(len(pos_scores)*len(decoy_scores))

print("NEGATIVE CONTROL / DECOY DISCRIMINATION")
print(f"  Mann-Whitney U = {mwu_stat:.1f},  p = {mwu_p:.2e}")
print(f"  KS statistic   = {ks_stat:.4f},  p = {ks_p:.2e}")
print(f"  Effect size r  = {effect_r:.4f}")

In [ ]:
# ── CELL 10: PCA of chemical feature space ───────────────────────────────────

pca = PCA(n_components=2, random_state=42)
Xpca_all = np.vstack([X_raw, X_lib_raw[:500]])
Xpca = pca.fit_transform(scaler.transform(Xpca_all))

pc1v = pca.explained_variance_ratio_[0]*100
pc2v = pca.explained_variance_ratio_[1]*100
print(f"PC1 variance explained: {pc1v:.1f}%")
print(f"PC2 variance explained: {pc2v:.1f}%")

In [ ]:
# ── CELL 11: RESULTS BLOCK — copy and paste to Claude ───────────────────────

print("="*70)
print("RESULTS BLOCK — PASTE THIS TO CLAUDE")
print("="*70)

rf2  = RandomForestRegressor(n_estimators=500,max_features="sqrt",min_samples_leaf=2,random_state=42)
gbr2 = GradientBoostingRegressor(n_estimators=300,learning_rate=0.05,max_depth=2,random_state=42)
yrf  = cross_val_predict(rf2,  X, y, cv=LeaveOneOut())
ygbr = cross_val_predict(gbr2, X, y, cv=LeaveOneOut())

for name,pred in [("RF",yrf),("GBR",ygbr)]:
    r2  = r2_score(y,pred)
    mae = mean_absolute_error(y,pred)
    ci_r2  = bootstrap_ci(y,pred,r2_score)
    ci_mae = bootstrap_ci(y,pred,mean_absolute_error)
    print(f"\n{name} LOO-CV R2  = {r2:.4f}  (95% CI: {ci_r2[0]:.4f} \u2013 {ci_r2[1]:.4f})")
    print(f"{name} LOO-CV MAE = {mae:.4f}  (95% CI: {ci_mae[0]:.4f} \u2013 {ci_mae[1]:.4f})")

print(f"\nTotal screened    = {len(full_lib):,}")
print(f"High-affinity+div = {len(top_idx):,}")
print(f"Top candidate     = {full_lib[top_sorted[0]]} (ens={s_ens[top_sorted[0]]:.4f})")
print(f"Mann-Whitney p    = {mwu_p:.2e}")
print(f"KS p              = {ks_p:.2e}")
print(f"Effect size r     = {effect_r:.4f}")
print(f"PCA PC1           = {pc1v:.1f}%  PC2 = {pc2v:.1f}%")
print("="*70)

In [ ]:
# ── CELL 12: Figure 1 — SCI Energy Profiles ─────────────────────────────

fig,axes = plt.subplots(1,2,figsize=(10,4),sharey=True)
fig.subplots_adjust(wspace=0.08)

ls_map = {"A":"-","G":"-","C":"--","T":"-.","U":":"}
col_pur = {"A":TEAL,"G":BLUE}
col_pyr = {"C":AMBER,"T":RED,"U":PURPLE}

for nt,col in col_pur.items():
    e = np.log10(np.abs(SCI_PROFILES[nt])+1e-30)
    axes[0].plot(r_vals,e,color=col,lw=2,ls=ls_map[nt],label=f"{nt} (q={FUNCTIONAL_CHARGE[nt]})")
for nt,col in col_pyr.items():
    e = np.log10(np.abs(SCI_PROFILES[nt])+1e-30)
    axes[1].plot(r_vals,e,color=col,lw=2,ls=ls_map[nt],label=f"{nt} (q={FUNCTIONAL_CHARGE[nt]})")
for ax,title in zip(axes,["Purines (A, G)","Pyrimidines (C, T, U)"]):
    ax.set_xlabel("Distance (nm)"); ax.set_xlim(0.1,5.0)
    ax.legend(frameon=False); ax.set_title(title,fontweight="bold")
    ax.axhline(0,color=GRAY,lw=0.5,ls="--",alpha=0.5)
axes[0].set_ylabel("log\u2081\u2080|E_SCI| (normalised)")
plt.suptitle("Figure 1. SCI binding energy profiles — rank G > C > A > T \u2248 U",
             fontsize=9,style="italic")
plt.savefig("Figure1_SCI_Profiles.png",dpi=300); plt.show()
print("Figure 1 saved.")

In [ ]:
# ── CELL 13: Figure 2 — Model Performance ───────────────────────────────

cat_colors = {"Universal":BLUE,"DNA":TEAL,"RNA":AMBER,"Hybrid":PURPLE,"Negative":RED}
cat_markers= {"Universal":"o","DNA":"s","RNA":"^","Hybrid":"D","Negative":"X"}

fig,axes = plt.subplots(1,2,figsize=(10,5))
fig.subplots_adjust(wspace=0.35)

for ax,(pred,title,r2_v,mae_v,ci_r2_v,ci_mae_v) in zip(axes,[
    (y_pred_rf, "Random Forest","0.140","0.120","\u22120.117\u20130.382","0.050\u20130.213"),
    (y_pred_gbr,"Gradient Boosting","0.473","0.084","0.281\u20130.831","0.027\u20130.160")
]):
    seen=set()
    for i,(yv,pv) in enumerate(zip(y,pred)):
        cat=cats[i]; lbl=cat if cat not in seen else ""
        ax.scatter(yv,pv,color=cat_colors[cat],marker=cat_markers[cat],s=60,
                   label=lbl,zorder=3,edgecolors="white",lw=0.5)
        seen.add(cat)
    lo,hi=min(y)-0.05,max(y)+0.05
    ax.plot([lo,hi],[lo,hi],color=GRAY,lw=1,ls="--",alpha=0.7)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel("Actual binding score"); ax.set_ylabel("LOO-CV predicted score")
    ax.set_title(title,fontweight="bold")
    ax.text(0.04,0.97,f"R\u00b2 = {r2_v}\nMAE = {mae_v}\n95% CI R\u00b2: {ci_r2_v}",
            transform=ax.transAxes,fontsize=8,va="top",
            bbox=dict(boxstyle="round,pad=0.4",fc=LGRAY,ec="#CCCCCC",lw=0.8))
handles=[mpatches.Patch(color=cat_colors[c],label=c)
         for c in ["Universal","DNA","RNA","Hybrid","Negative"]]
axes[0].legend(handles=handles,frameon=False,fontsize=8,loc="lower right")
plt.suptitle("Figure 2. LOO-CV performance with bootstrapped 95% CI (B = 2,000)",
             fontsize=9,style="italic")
plt.savefig("Figure2_Model_Performance.png",dpi=300); plt.show()
print("Figure 2 saved.")

In [ ]:
# ── CELL 14: Figures 3–5 (Feature Importance, Screening, PCA+Decoy) ───────
# Run generate_figures.py or see repository scripts/generate_figures.py
# for the complete publication-quality versions of Figures 3, 4, and 5.

# Quick preview — Feature importance (Figure 3)
fig,ax=plt.subplots(figsize=(7,5))
top15=feat_df.head(15)
yp=np.arange(len(top15))
ax.barh(yp,top15["perm_mean"].values[::-1],
        xerr=top15["perm_std"].values[::-1],
        color=[BLUE if m>0.05 else TEAL if m>0.02 else GRAY
               for m in top15["perm_mean"].values],
        error_kw=dict(elinewidth=0.8,capsize=2,ecolor="#555"),height=0.65)
ax.set_yticks(yp); ax.set_yticklabels(top15["feature"].values[::-1],fontsize=8.5)
ax.set_xlabel("Permutation importance (mean \u00b1 SD)")
ax.set_title("Figure 3A. Top 15 features — permutation importance",fontweight="bold")
plt.tight_layout(); plt.savefig("Figure3_Feature_Importance_preview.png",dpi=150)
plt.show(); print("Figure 3 preview saved.")

In [ ]:
# ── CELL 15: Export results CSV ─────────────────────────────────────────

# Save top 10 candidates
df_top10.to_csv("top10_candidates.csv", index=False)
print("Saved: top10_candidates.csv")

# Save training data
df_train.to_csv("training_aptamers.csv", index=False)
print("Saved: training_aptamers.csv")

# Save full top candidates (top 100)
rows=[]
for i in top_sorted[:100]:
    rows.append({"sequence":full_lib[i],"type":lib_types[i],
                 "ens_score":round(s_ens[i],4),"struct_score":round(ss[i],3),
                 "composite":round(comp[i],4)})
pd.DataFrame(rows).to_csv("top100_candidates.csv",index=False)
print("Saved: top100_candidates.csv")
print("\nAll outputs saved. Download from the Files panel (left sidebar in Colab).")